In [1]:
# =============================
# REBUILD CLEAN MASTER (UNSCALED)
# =============================

import pandas as pd

performance_df = pd.read_csv("../data/processed/performance_data.csv")
market_df = pd.read_csv("../data/processed/market_value_data.csv")
sentiment_df = pd.read_csv("../data/processed/sentiment_data.csv")
injury_df = pd.read_csv("../data/processed/injury_data.csv")

clean_df = market_df.copy()

clean_df = clean_df.merge(performance_df, on="player_name", how="left")
clean_df = clean_df.merge(sentiment_df, on="player_name", how="left")
clean_df = clean_df.merge(injury_df, on="player_name", how="left")

print("Clean rebuilt:", clean_df.shape)

Clean rebuilt: (268, 10)


In [2]:
import numpy as np

clean_df["goal_conversion_rate"] = (
    clean_df["goals"] / clean_df["shots"].replace(0, np.nan)
).fillna(0)

clean_df["minutes_per_goal"] = (
    clean_df["total_minutes"] / clean_df["goals"].replace(0, np.nan)
).fillna(clean_df["total_minutes"])

clean_df["performance_index"] = (
    0.4 * clean_df["goals"]
    + 0.3 * clean_df["shots"]
    + 0.3 * clean_df["passes"]
)

clean_df["injury_risk_score"] = (
    clean_df["total_days_out"] / (clean_df["total_minutes"] + 1)
)

In [3]:
clean_df.to_csv("../data/processed/final_dataset.csv", index=False)

print("✅ final_clean_dataset.csv fixed (UNSCALED)")

✅ final_clean_dataset.csv fixed (UNSCALED)


In [4]:
from sklearn.preprocessing import StandardScaler

model_df = clean_df.copy()

scale_cols = [
    c for c in model_df.select_dtypes(include="number").columns
    if c != "market_value"
]

scaler = StandardScaler()
model_df[scale_cols] = scaler.fit_transform(model_df[scale_cols])

In [5]:
X = model_df.drop(columns=["player_name", "market_value"])
y = model_df["market_value"]

X.to_csv("../data/processed/model_features_scaled.csv", index=False)
y.to_csv("../data/processed/model_target.csv", index=False)

print("✅ Scaled model files ready")

✅ Scaled model files ready
